# Trainable Intervention Recipe

Some interventions are not fixed ablations or swaps: they are learned parameters. This recipe learns a small additive vector at one activation site by replaying a captured TorchLens graph and backpropagating through the replayed output. The loop retains the replay graph because the captured upstream graph is reused while only the intervention vector is optimized.


In [ ]:
from __future__ import annotations

from typing import Any

import torch
from torch import nn

import torchlens as tl


class TinyRegressor(nn.Module):
    """Small ReLU regressor used for trainable intervention replay."""

    def __init__(self) -> None:
        """Initialize deterministic layers."""

        super().__init__()
        self.hidden = nn.Linear(4, 4)
        self.head = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the regressor.

        Parameters
        ----------
        x:
            Batch of four-dimensional inputs.

        Returns
        -------
        torch.Tensor
            Two-dimensional output.
        """

        return self.head(torch.relu(self.hidden(x)))


torch.manual_seed(5)
model = TinyRegressor().eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
x = torch.randn(3, 4)
target = torch.tensor([[0.5, -0.2], [0.2, 0.1], [-0.1, 0.4]])
base_log = tl.log_forward_pass(model, x, vis_opt="none", intervention_ready=True)
base_loss = torch.nn.functional.mse_loss(base_log.layer_list[-1].activation.detach(), target)
print("base loss", float(base_loss))

In [ ]:
delta = torch.nn.Parameter(torch.zeros(4))
optimizer = torch.optim.SGD([delta], lr=0.25)
loss_history: list[float] = []


def add_trainable_delta(activation: torch.Tensor, *, hook: Any) -> torch.Tensor:
    """Add the learned intervention vector to a hidden activation.

    Parameters
    ----------
    activation:
        Activation intercepted by TorchLens.
    hook:
        TorchLens hook context.

    Returns
    -------
    torch.Tensor
        Shifted activation.
    """

    return activation + delta


for step in range(15):
    optimizer.zero_grad()
    trained_log = base_log.fork(f"train_step_{step}")
    trained_log.attach_hooks(tl.func("relu"), add_trainable_delta).replay()
    prediction = trained_log.layer_list[-1].activation
    loss = torch.nn.functional.mse_loss(prediction, target)
    loss.backward(retain_graph=True)
    optimizer.step()
    loss_history.append(float(loss.detach()))

print(loss_history)
print("learned delta", delta.detach())
assert loss_history[-1] < loss_history[0]